# Repair experiment metrics and assembly viewer

For a single focused experiment such as `long_control_retry`. **Run all cells** after setting `RUN_DIR` below. This notebook reads saved files and does not train, change gates, or run model inference.

The overfit model trains on 16 patterns; the main contact model trains on the full training split. Their evaluations remain separate. Validation observations repeat each pattern under three sampling/pose seeds. Distances use normalized units. Geometric success and confidence acceptance are shown separately.

Required packages: `numpy pandas matplotlib`. Optional: `ipywidgets` for sample selectors and `plotly` for rotatable 3D views. Saved `example_*.npz` files are needed for assembly visualization; JSON alone supports the metrics.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RUN_DIR = Path.home() / "reassembly_v2/repair_gate_experiments/long_control_retry"
# Optional old experiment root for side-by-side comparisons:
BASELINE_DIR = None  # Path.home() / "reassembly_v2/repair_v3/experiments/revised__resampled_contrastive/seed42"
THRESHOLDS = [0.01, 0.02, 0.03]

if not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Set RUN_DIR to the experiment directory, containing gate/ and contact_validation/: {RUN_DIR}")

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def load_jsonl(path):
    path = Path(path)
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()] if path.exists() else []

EVALUATIONS = {"Overfit model": "gate/overfit", "Main model: training": "gate/training", "Main model: validation": "contact_validation"}
records = {}
reports = {}
for label, relative in EVALUATIONS.items():
    rows = load_jsonl(RUN_DIR / relative / "examples.jsonl")
    records[label] = rows
    path = RUN_DIR / relative / "evaluation.json"
    if path.exists(): reports[label] = load_json(path)
    print(f"{label}: {len(rows)} observations, {len({r['pattern_id'] for r in rows})} patterns")
if not any(records.values()): raise FileNotFoundError("No examples.jsonl found in the expected evaluation directories.")


## Gate and training configuration

The official gate keeps its original threshold. Threshold sweeps later in this notebook are diagnostic. Fixed-16 also requires no low-confidence results; the robustness/training checks use geometric success.

In [ ]:
gate_path = RUN_DIR / "gate/contact_gate.json"
if gate_path.exists():
    gate = load_json(gate_path)
    display(pd.DataFrame([{"check": k, "passed": v} for k, v in gate["checks"].items()]))
    display(pd.DataFrame([{"condition": k, "success_rate": v} for k, v in gate["robustness"].items()]))
    print("Training geometric success:", gate["training_success"], "Overall gate passed:", gate["passed"])
else: print("Gate report is not available yet.")
for phase in ("overfit", "contacts"):
    path = RUN_DIR / phase / "config.resolved.json"
    if path.exists():
        cfg = load_json(path)
        print(phase, {k: cfg["train"].get(k) for k in ("overfit_updates", "max_updates", "batch_size", "grad_accum_steps", "learning_rate")})


## Every observation and recorded metric

`raw_metrics` contains flattened rows including segmentation, pair diagnostics, candidate scores and representation metrics when saved. `metrics` below is the compact per-assembly table. Export CSV only if desired. Missing poses stay in success denominators; distance means exclude them. Worst-part distance is more informative than pooled part statistics, which include the fixed reference fragment.

In [ ]:
flat_rows = []
compact_rows = []
for label, rows in records.items():
    for row in rows:
        flat_rows.append({"evaluation": label, **row})
        parts = row.get("per_part_chamfer")
        rotations = row.get("rotation_deg")
        compact_rows.append({"evaluation": label, **{k: row.get(k) for k in
            ("pattern_id", "source_id", "mode", "seed", "condition", "pieces", "band", "cut_family", "status", "failed", "success", "confidence", "whole_chamfer", "contact_rms", "reason")},
            "max_part_chamfer": max(parts) if parts else np.nan,
            "max_rotation_deg": max(rotations) if rotations else np.nan,
            "candidate_covered": row.get("candidate_coverage", {}).get("candidate_oracle_success", False),
            "orientation_score": row.get("diagnostics", {}).get("orientation_score", np.nan),
            "orientation_reliable_fraction": row.get("diagnostics", {}).get("orientation_reliable_fraction", np.nan),
            "selected_proposal_types": row.get("diagnostics", {}).get("selected_proposal_types", []),
            "rotation_flip_indicator": not row["success"] and max(rotations or [0]) > 170})
raw_metrics = pd.json_normalize(flat_rows)
metrics = pd.DataFrame(compact_rows)
metrics["valid_candidate_lost"] = metrics["candidate_covered"].astype(bool) & ~metrics["success"].astype(bool)
metrics["accepted_success"] = metrics["success"].astype(bool) & metrics["status"].eq("ok")
display(metrics)
print("All recorded fields:", list(raw_metrics.columns))
# Optional: raw_metrics.to_csv(RUN_DIR / "all_observation_metrics.csv", index=False)


In [ ]:
summary = metrics.groupby(["evaluation", "mode", "seed", "condition"], dropna=False).agg(
    observations=("success", "size"), patterns=("pattern_id", "nunique"),
    geometric_successes=("success", "sum"), geometric_rate=("success", "mean"),
    accepted_successes=("accepted_success", "sum"), accepted_rate=("accepted_success", "mean"),
    candidate_coverage=("candidate_covered", "mean"),
    whole_mean=("whole_chamfer", "mean"), whole_median=("whole_chamfer", "median"),
    worst_part_mean=("max_part_chamfer", "mean"), worst_part_median=("max_part_chamfer", "median"),
    confidence_mean=("confidence", "mean"))
display(summary)
# Full official summaries include pooled fragment mean/median/P90 and source-level metrics.
for label, report in reports.items():
    print(label)
    display(pd.json_normalize([{"observation_group": k, **v} for k, v in report.get("summaries", {}).items()]))


## Learning curves for the two independent models

Recorded validation metrics during the overfit run refer to the fixed training subset; during the main run they refer to validation. These curves describe periodic checkpoints, whereas final evaluation uses the selected best checkpoint and declared evaluation seeds.

In [ ]:
histories = {}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for phase, title in (("overfit", "Overfit model"), ("contacts", "Main contact model")):
    rows = load_jsonl(RUN_DIR / phase / "history.jsonl")
    if not rows: continue
    frame = pd.json_normalize(rows).sort_values("update")
    histories[phase] = frame
    for ax, field in zip(axes.flat, ["validation.assembly_success", "validation.matching_top1_recall", "validation.retrieval_top5", "validation.matching_mass", "validation.matching_localization", "validation.segmentation"]):
        if field in frame: ax.plot(frame["update"], frame[field], label=title)
        ax.set(title=field.removeprefix("validation."), xlabel="Optimizer update")
        ax.grid(alpha=.2)
for ax in axes.flat:
    if ax.lines: ax.legend()
fig.tight_layout()
plt.show()
for phase, frame in histories.items():
    print(phase, "all logged learning metrics")
    display(frame.tail())


## Error distributions and confidence

Histograms include all produced poses, even geometrically incorrect and low-confidence results. Missing-pose counts are displayed separately. The CDF curve uses **all observations** in its denominator, so it cannot hide failed solves. Confidence >= 0.1 alone does not establish geometric correctness.

In [ ]:
main = metrics[metrics["mode"].eq("samples_poses") & metrics["condition"].eq("contact_only")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for label in EVALUATIONS:
    frame = main[main["evaluation"].eq(label)]
    if frame.empty: continue
    vals = frame["max_part_chamfer"].dropna().sort_values().to_numpy()
    axes[0].hist(vals, bins=30, alpha=.4, label=label)
    axes[1].step(vals, np.arange(1, len(vals)+1)/len(frame), where="post", label=label)
    axes[2].scatter(frame["max_part_chamfer"], frame["confidence"], s=12, alpha=.4, label=label)
    print(label, "no pose:", int(frame["whole_chamfer"].isna().sum()), "/", len(frame))
for t in THRESHOLDS:
    axes[0].axvline(t, ls="--", color="grey", alpha=.5)
    axes[1].axvline(t, ls="--", color="grey", alpha=.5)
axes[0].set(title="Worst-fragment errors", xlabel="Normalized Chamfer", ylabel="Observations")
axes[1].set(title="Worst-fragment error CDF", xlabel="Normalized Chamfer", ylabel="Fraction of all observations", ylim=(0, 1))
axes[2].set(title="Confidence versus error", xlabel="Worst-fragment Chamfer", ylabel="Confidence")
axes[2].axhline(.1, color="black", ls="--")
for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.2)
fig.tight_layout(); plt.show()


## Success at 0.01, 0.02 and 0.03

A success requires **whole-object AND every fragment** to meet the chosen threshold. Failed solves remain failures. The confidence-aware column also requires the existing `ok` status. This does not modify the official gate. Observations of the same pattern are correlated; counts are not counts of distinct source objects.

In [ ]:
def threshold_table(frame, thresholds=THRESHOLDS):
    output = []
    for (label, mode), group in frame.groupby(["evaluation", "mode"]):
        for threshold in thresholds:
            passed = group["whole_chamfer"].le(threshold) & group["max_part_chamfer"].le(threshold)
            output.append(dict(evaluation=label, mode=mode, threshold=threshold, attempts=len(group),
                successes=int(passed.sum()), success_rate=float(passed.mean()),
                accepted_successes=int((passed & group["status"].eq("ok")).sum())))
    return pd.DataFrame(output)
sweep = threshold_table(metrics[metrics["condition"].eq("contact_only")])
display(sweep)
plot_data = sweep[sweep["mode"].eq("samples_poses")].pivot(index="threshold", columns="evaluation", values="success_rate")
plot_data.plot.bar(figsize=(10, 4), ylim=(0, 1), ylabel="Geometric success rate", title="Threshold sensitivity (all attempts)")
plt.tight_layout(); plt.show()
# Sanity check: the strict diagnostic threshold reproduces the saved geometric labels.
strict = metrics["whole_chamfer"].le(.01) & metrics["max_part_chamfer"].le(.01)
assert np.array_equal(strict.to_numpy(), metrics["success"].astype(bool).to_numpy()), "Saved labels differ from the .01 metric: inspect the inputs."


## Which patterns and fragment counts fail?

The following tables separate confidence, geometric error and missing candidates. Pattern stability shows successes across repeated seeds, not distinct objects. Source-macro rates weight each source equally.

In [ ]:
for field in ("pieces", "band", "cut_family"):
    display(main.groupby(["evaluation", field]).agg(attempts=("success", "size"), successes=("success", "sum"),
        geometric_rate=("success", "mean"), accepted_rate=("accepted_success", "mean"),
        candidate_coverage=("candidate_covered", "mean"), worst_part_median=("max_part_chamfer", "median")))
display(main.groupby(["evaluation", "status"]).size().rename("attempts").to_frame())
pattern_stability = main.groupby(["evaluation", "pattern_id"]).agg(attempts=("success", "size"), successes=("success", "sum"), worst_error=("max_part_chamfer", "max"))
display(pattern_stability.sort_values(["successes", "worst_error"], ascending=[True, False]))
source_rates = main.groupby(["evaluation", "source_id"])["success"].mean()
display(source_rates.groupby("evaluation").mean().rename("source_macro_success").to_frame())


## Orientation and candidate selection diagnostics

The `contact_only` condition does not use the learned scaffold prior. Optional contact orientation uses only fragment XYZ. Missing orientation columns mean the intervention was disabled in that run. A rotation error above 170 degrees on a failed reconstruction is a diagnostic indicator, not proof of planar symmetry. Candidate loss means a valid unrefined candidate existed but the final reconstruction failed; this can involve selection or refinement.

In [ ]:
orientation_summary = main.groupby("evaluation").agg(
    attempts=("success", "size"), geometric_successes=("success", "sum"),
    accepted_successes=("accepted_success", "sum"), flip_indicators=("rotation_flip_indicator", "sum"),
    valid_candidate_lost=("valid_candidate_lost", "sum"),
    normal_score_mean=("orientation_score", "mean"), reliable_mass_fraction_mean=("orientation_reliable_fraction", "mean"))
display(orientation_summary)
display(main[main["rotation_flip_indicator"] | main["valid_candidate_lost"]][[
    "evaluation", "pattern_id", "seed", "pieces", "status", "max_rotation_deg", "max_part_chamfer",
    "candidate_covered", "orientation_score", "orientation_reliable_fraction", "selected_proposal_types"]])


## Optional comparison with the earlier run

Set `BASELINE_DIR` in the setup cell. Aggregate rates use the rows saved in each run; ensure matching datasets, modes, conditions and seeds before interpreting differences as a controlled comparison.

In [ ]:
if BASELINE_DIR is not None:
    comparison = []
    for run_name, directory in (("Baseline", Path(BASELINE_DIR)), ("Current", RUN_DIR)):
        for label, relative in EVALUATIONS.items():
            rows = [r for r in load_jsonl(directory / relative / "examples.jsonl") if r["mode"] == "samples_poses" and r["condition"] == "contact_only"]
            if rows:
                comparison.append(dict(run=run_name, evaluation=label, attempts=len(rows),
                    geometric_success=sum(r["success"] for r in rows)/len(rows),
                    accepted_success=sum(r["success"] and r["status"] == "ok" for r in rows)/len(rows)))
    display(pd.DataFrame(comparison))
else: print("Set BASELINE_DIR to enable earlier-run comparison.")


## Training and validation sample assemblies

Each color is one fragment. Panels show the independently posed input, predicted reconstruction, and ground truth. Predicted and target panels share axis limits. The transform is `aligned = points @ rotation.T + translation`.

These are the bounded examples saved by evaluation (usually its first eight rows), **not a representative random sample**. A failed solve shows an empty prediction panel, never ground truth disguised as prediction. Select training or validation and the saved observation below. Plotly lets you rotate/zoom each panel; Matplotlib is the fallback. No GPU or checkpoint is needed.

Prediction and ground truth now start with the same camera and axis limits. Rotating one panel manually changes its view independently; reset both before judging apparent flips.

In [ ]:
samples = {}
for label, relative in EVALUATIONS.items():
    rows = records[label]
    entries = []
    for path in sorted((RUN_DIR / relative).glob("example_*.npz")):
        index = int(path.stem.split("_")[-1]) - 1
        if index < len(rows): entries.append((path, rows[index]))
    samples[label] = entries
    print(label, len(entries), "saved assemblies")

def sample_arrays(path):
    with np.load(path, allow_pickle=False) as archive:
        count = int(archive["mask"].sum())
        points, target = archive["points"][:count].copy(), archive["target"][:count].copy()
        predicted = None
        if "rotations" in archive and "translations" in archive:
            predicted = np.einsum("fni,fji->fnj", points, archive["rotations"][:count]) + archive["translations"][:count, None]
    return points, predicted, target

COLORS = ["#0072B2", "#D55E00", "#009E73"]
def show_assembly(label, index=0, interactive=True):
    if not samples.get(label):
        print("No saved example NPZ files for", label); return
    path, row = samples[label][index]
    points, predicted, target = sample_arrays(path)
    diagnostic = row.get("diagnostics", {})
    if "orientation_score" in diagnostic:
        display(pd.DataFrame([{k: diagnostic.get(k) for k in ("orientation_score", "orientation_reliable_fraction", "orientation_reliable_mass", "selected_proposal_types", "refinement_decisions")}]))
    display(pd.DataFrame([{k: row.get(k) for k in ("pattern_id", "mode", "seed", "pieces", "status", "success", "confidence", "whole_chamfer", "per_part_chamfer", "rotation_deg", "reason")}]))
    panels = [("Input fragments", points), ("Predicted assembly" if predicted is not None else "Prediction: no pose", predicted), ("Ground truth", target)]
    combined = np.concatenate([target.reshape(-1, 3)] + ([predicted.reshape(-1, 3)] if predicted is not None else []))
    center = (combined.max(0)+combined.min(0))/2
    radius = max(np.ptp(combined, axis=0).max()/2, .05)*1.05
    try:
        if not interactive: raise ImportError()
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        fig = make_subplots(rows=1, cols=3, specs=[[{"type":"scene"}]*3], subplot_titles=[p[0] for p in panels])
        for col, (_, arrays) in enumerate(panels, 1):
            if arrays is None: continue
            for part, cloud in enumerate(arrays):
                selected = cloud[::max(1, len(cloud)//700)]
                fig.add_trace(go.Scatter3d(x=selected[:,0], y=selected[:,1], z=selected[:,2], mode="markers",
                    marker=dict(size=2, color=COLORS[part]), name=f"Fragment {part}", legendgroup=str(part), showlegend=col==1), row=1, col=col)
        camera = dict(eye=dict(x=1.5, y=1.5, z=1.2), up=dict(x=0, y=0, z=1))
        fig.update_scenes(aspectmode="data", camera=camera)
        for scene in ("scene2", "scene3"):
            fig.update_layout(**{scene: dict(aspectmode="cube", camera=camera, **{axis:dict(range=[float(center[k]-radius),float(center[k]+radius)]) for k, axis in enumerate(("xaxis","yaxis","zaxis"))})})
        fig.update_layout(height=480, width=1150, title=row["pattern_id"])
        display(fig)
    except ImportError:
        fig = plt.figure(figsize=(15, 5))
        for col, (title, arrays) in enumerate(panels, 1):
            ax = fig.add_subplot(1, 3, col, projection="3d")
            if arrays is not None:
                for part, cloud in enumerate(arrays):
                    selected = cloud[::max(1, len(cloud)//700)]
                    ax.scatter(*selected.T, s=2, alpha=.6, color=COLORS[part], label=f"Fragment {part}")
            if col > 1:
                ax.set_xlim(center[0]-radius,center[0]+radius); ax.set_ylim(center[1]-radius,center[1]+radius); ax.set_zlim(center[2]-radius,center[2]+radius)
            ax.view_init(elev=20, azim=-55)
            ax.set(title=title, xlabel="X", ylabel="Y", zlabel="Z"); ax.set_box_aspect((1,1,1))
        fig.suptitle(row["pattern_id"]); fig.tight_layout(); plt.show()


In [ ]:
# Always show a training and validation example, even without widget support.
for label in ("Main model: training", "Main model: validation"):
    if samples[label]: show_assembly(label, 0)
try:
    import ipywidgets as widgets
    available = [label for label, entries in samples.items() if entries]
    if available:
        split = widgets.Dropdown(options=available, value=available[0], description="Evaluation:", style={"description_width":"initial"})
        example = widgets.Dropdown(description="Assembly:", style={"description_width":"initial"}, layout=widgets.Layout(width="85%"))
        output = widgets.Output()
        def choose_options(*_):
            example.options = [(f"{i+1}: {r['pattern_id']} | {r['mode']}/{r['seed']} | success={r['success']} | {r['status']}", i) for i, (_, r) in enumerate(samples[split.value])]
        def render(*_):
            if example.value is None: return
            with output:
                output.clear_output(wait=True)
                show_assembly(split.value, example.value)
        split.observe(choose_options, names="value")
        example.observe(render, names="value")
        choose_options()
        display(widgets.VBox([split, example, output])); render()
except ImportError:
    print("Optional ipywidgets unavailable. Call show_assembly('Main model: training', 1) to view another sample.")


## Inspect any saved sample manually

Examples: `show_assembly("Main model: training", 3)` or `show_assembly("Main model: validation", 2)`. Indexes start at zero. For additional assemblies beyond the saved examples, new inference/export is necessary; existing JSON does not store full pose matrices for every row.

To share this analysis, copy the experiment directory including the two training histories, gate report, all `evaluation.json` and `examples.jsonl` files, and the `example_*.npz` files. Full checkpoints and prepared data are not required for this notebook.